# SOM Topology Graph vs. DS2LSOM Density Graph

Both graphs live on the same k prototypes, but encode different structure:

- **SOM graph** (`quantizer_.som_`): edges = topological neighbors during training (grid adjacency)
- **DS2LSOM graph** (`graph_`): edges = prototypes that share samples as co-BMU (data-driven)

Comparing them reveals which edges are cluster boundaries vs. data bridges.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

from ds2l_som.ds2lsom import DS2LSOM

digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
y = digits.target

model = DS2LSOM(
    n_prototypes=100,
    threshold=10,
    method="som",
    model_args={"init": {"sigma_end": 1.0, "random_state": 42}},
)
model.fit(X)
print(
    f"Prototypes: {model.n_prototypes_}  Clusters: {len(set(model.labels_[model.labels_ >= 0]))}"
)

## Build node mapping and edge sets


In [ ]:
som = model.quantizer_

# neurons_[i] = 2D grid position of prototype i
idx_to_grid = {i: pos for i, pos in enumerate(som.neurons_)}
grid_to_idx = {pos: i for i, pos in idx_to_grid.items()}

# Translate SOM edges (grid tuples) → integer indices
som_edges = frozenset(
    frozenset([grid_to_idx[u], grid_to_idx[v]])
    for u, v in som.som_.edges()
    if u in grid_to_idx and v in grid_to_idx
)
ds2l_edges = frozenset(frozenset([u, v]) for u, v in model.graph_.edges())

both = som_edges & ds2l_edges
only_som = som_edges - ds2l_edges
only_ds2l = ds2l_edges - som_edges

print(f"SOM edges:     {len(som_edges):4d}")
print(f"DS2LSOM edges: {len(ds2l_edges):4d}")
print(f"Both:          {len(both):4d}")
print(f"Only SOM:      {len(only_som):4d}")
print(f"Only DS2LSOM:  {len(only_ds2l):4d}")

## Edge statistics


In [ ]:
def label_of(node):
    return model.graph_.nodes.get(node, {}).get("label", -1)


def crossing(edge_set):
    """Fraction of edges that cross a cluster boundary."""
    if not edge_set:
        return float("nan")
    cross = sum(1 for e in edge_set if label_of(list(e)[0]) != label_of(list(e)[1]))
    return cross / len(edge_set)


stats = pd.DataFrame(
    [
        {
            "Edge type": "Both",
            "Count": len(both),
            "Boundary crossing": f"{crossing(both):.1%}",
        },
        {
            "Edge type": "Only SOM",
            "Count": len(only_som),
            "Boundary crossing": f"{crossing(only_som):.1%}",
        },
        {
            "Edge type": "Only DS2LSOM",
            "Count": len(only_ds2l),
            "Boundary crossing": f"{crossing(only_ds2l):.1%}",
        },
    ]
).set_index("Edge type")

print(stats.to_string())
print()
print("Interpretation:")
print("  Both       – topological neighbor AND shared data space")
print("  Only SOM   – adjacent in grid, but no shared samples → cluster boundary")
print("  Only DS2LSOM – shared samples despite non-adjacent position → data bridge")

## Visualization

Node positions = SOM grid coordinates. Node size = local density. Node color = DS2LSOM cluster label.


In [ ]:
# Build shared layout from grid coordinates
nodes_in_ds2l = set(model.graph_.nodes)
pos = {
    i: (float(x), float(y)) for i, (x, y) in idx_to_grid.items() if i in nodes_in_ds2l
}

cluster_labels = {n: model.graph_.nodes[n]["label"] for n in nodes_in_ds2l}
unique_labels = sorted(set(cluster_labels.values()))
cmap = plt.cm.get_cmap("tab10", len(unique_labels))
label_to_color = {lbl: cmap(i) for i, lbl in enumerate(unique_labels)}
node_colors = [label_to_color[cluster_labels[n]] for n in sorted(nodes_in_ds2l)]

densities = np.array([model.graph_.nodes[n]["density"] for n in sorted(nodes_in_ds2l)])
node_sizes = 50 + 800 * (
    densities / densities.max() if densities.max() > 0 else densities
)
node_list = sorted(nodes_in_ds2l)


def draw(ax, edge_set, title, edge_color="#555555"):
    nx.draw_networkx_nodes(
        model.graph_,
        pos,
        nodelist=node_list,
        node_color=node_colors,
        node_size=node_sizes,
        ax=ax,
        alpha=0.9,
    )
    edges = [
        (list(e)[0], list(e)[1])
        for e in edge_set
        if list(e)[0] in pos and list(e)[1] in pos
    ]
    nx.draw_networkx_edges(
        model.graph_,
        pos,
        edgelist=edges,
        edge_color=edge_color,
        width=1.2,
        ax=ax,
        alpha=0.7,
    )
    ax.set_title(title)
    ax.axis("off")


fig, axes = plt.subplots(1, 3, figsize=(18, 6))

draw(axes[0], som_edges, f"SOM graph ({len(som_edges)} edges)", edge_color="#2196F3")
draw(
    axes[1],
    ds2l_edges,
    f"DS2LSOM graph ({len(ds2l_edges)} edges)",
    edge_color="#E53935",
)

# Overlap plot: three colors
nx.draw_networkx_nodes(
    model.graph_,
    pos,
    nodelist=node_list,
    node_color=node_colors,
    node_size=node_sizes,
    ax=axes[2],
    alpha=0.9,
)
for edge_set, color, label in [
    (both, "#888888", "both"),
    (only_som, "#2196F3", "only SOM"),
    (only_ds2l, "#E53935", "only DS2LSOM"),
]:
    edges = [
        (list(e)[0], list(e)[1])
        for e in edge_set
        if list(e)[0] in pos and list(e)[1] in pos
    ]
    nx.draw_networkx_edges(
        model.graph_,
        pos,
        edgelist=edges,
        edge_color=color,
        width=1.5,
        ax=axes[2],
        alpha=0.8,
        label=label,
    )
axes[2].set_title("Edge overlap")
axes[2].axis("off")
axes[2].legend(loc="lower right", fontsize=9)

plt.suptitle(
    "SOM topology vs. DS2LSOM density graph  |  color = cluster  |  size = density",
    fontsize=11,
)
plt.tight_layout()
plt.show()